In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze_table = "crypto_project.crypto.bronze_crypto_market"
silver_table = "crypto_project.crypto.silver_crypto_market"

df_bronze = spark.table(bronze_table)

# 1. Derive market_date from source_file path (e.g. s3://.../raw/date=2026-11-15)
df_parsed = df_bronze.withColumn(
    "market_date",
    F.to_date(F.regexp_extract(F.col("source_file"), r"date=(\d{4}-\d{2}-\d{2})", 1))
)

# 2. Data Quality Filtering: valid ID, positive price, and market_cap > 0
df_clean = (
    df_parsed
    .filter(
        F.col("id").isNotNull()
        & F.col("market_date").isNotNull()
        & F.col("current_price").isNotNull()
        & (F.col("current_price") > 0)
        & F.col("market_cap").isNotNull()
        & (F.col("market_cap") > 0)
    )
    .select(
        F.col("id").alias("coin_id"),
        F.upper(F.col("symbol")).alias("symbol"),
        F.col("name"),
        F.col("market_date"),
        F.col("current_price").cast("double").alias("price_usd"),
        F.col("market_cap").cast("double").alias("market_cap"),
        F.col("total_volume").cast("double").alias("total_volume"),
        F.col("high_24h").cast("double").alias("high_24h"),
        F.col("low_24h").cast("double").alias("low_24h"),
        F.col("price_change_percentage_24h").cast("double").alias("price_change_24h_pct"),
        F.col("market_cap_rank").cast("int").alias("market_cap_rank"),
        F.col("ingested_at")
    )
)

# 3. Deduplicate: keep latest record per (coin_id, market_date)
dedup_window = Window.partitionBy("coin_id", "market_date").orderBy(F.col("ingested_at").desc())
df_deduped = (
    df_clean
    .withColumn("rn", F.row_number().over(dedup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# 4. Incremental MERGE into Silver (preserves full history, avoids duplicates)
if not spark.catalog.tableExists(silver_table):
    df_deduped.write.format("delta").saveAsTable(silver_table)
else:
    DeltaTable.forName(spark, silver_table).alias("t").merge(
        df_deduped.alias("s"),
        "t.coin_id = s.coin_id AND t.market_date = s.market_date"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

print("Silver layer updated: all coins with market_cap > 0 preserved!")

Silver layer updated: all coins with market_cap > 0 preserved!
